In [ ]:
from __future__ import annotations

import os
import shutil
import csv
import pickle
import subprocess
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
from Bio import SeqIO

In [ ]:
# ============================================================
# Project / paths
# ============================================================

def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()
PROJECT_ROOT = REPOSITORY_ROOT / "AA"
DATA_ROOT = PROJECT_ROOT / "data"

SIM_DIR = DATA_ROOT / "sim"
MATRIX_DIR = DATA_ROOT / "matrices"
BLAST_DB_DIR = DATA_ROOT / "blastdb"
MANIFEST_PATH = DATA_ROOT / "manifests" / "simulation_manifest.csv"

MAKEBLASTDB_EXE = Path(
    os.environ.get("MAKEBLASTDB_EXE")
    or shutil.which("makeblastdb")
    or "makeblastdb"
)
BLASTP_EXE = Path(
    os.environ.get("BLASTP_EXE")
    or shutil.which("blastp")
    or "blastp"
)


# ============================================================
# BLAST / matrix parameters
# ============================================================

SEQTYPE = "prot"
BLAST_TASK = "blastp"

EVALUE = 10.0
MAX_TARGET_SEQS = 1_000_000
MAX_HSPS = 1
NUM_THREADS = 4

EPS = 1e-12

# False: keep existing matrix outputs
# True: regenerate existing matrix outputs
OVERWRITE = False

In [ ]:
@dataclass
class MatrixRow:
    tag: str
    fasta_path: str
    n_sequences: int

    blast_tsv_path: str
    pickle_path: str
    npz_path: str

    status: str
    message: str


def ensure_directories() -> None:
    MATRIX_DIR.mkdir(parents=True, exist_ok=True)
    BLAST_DB_DIR.mkdir(parents=True, exist_ok=True)


def check_blast_executables() -> None:
    if not MAKEBLASTDB_EXE.exists():
        raise FileNotFoundError(
            f"makeblastdb was not found: {MAKEBLASTDB_EXE}"
        )

    if not BLASTP_EXE.exists():
        raise FileNotFoundError(
            f"blastp was not found: {BLASTP_EXE}"
        )


def read_simulation_manifest() -> list[dict]:
    rows: list[dict] = []

    with MANIFEST_PATH.open(newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row.get("status", "") in {"ok", "skipped_existing"}:
                rows.append(row)

    return rows


def resolve_fasta_path(
    row: dict,
) -> Path:
    manifest_path = Path(row["fasta_path"]).expanduser()

    if manifest_path.is_absolute():
        candidates = [manifest_path]
    else:
        candidates = [PROJECT_ROOT / manifest_path]

    # Backward-compatible fallback for manifests that only encode the tag.
    candidates.append(
        SIM_DIR / f"{row['tag']}.fa"
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"FASTA file was not found for tag={row['tag']}. "
        f"Checked: {checked}"
    )


def read_fasta_ids(fasta_path: Path) -> list[str]:
    return [
        record.id
        for record in SeqIO.parse(str(fasta_path), "fasta")
    ]


def make_blast_db(
    fasta_path: Path,
    db_prefix: Path,
) -> None:
    db_pin = db_prefix.with_suffix(".pin")

    if db_pin.exists() and not OVERWRITE:
        return

    command = [
        str(MAKEBLASTDB_EXE),
        "-in",
        str(fasta_path),
        "-dbtype",
        SEQTYPE,
        "-out",
        str(db_prefix),
    ]

    subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )


def run_self_blast(
    fasta_path: Path,
    db_prefix: Path,
    blast_tsv_path: Path,
) -> None:
    if blast_tsv_path.exists() and not OVERWRITE:
        return

    command = [
        str(BLASTP_EXE),
        "-query",
        str(fasta_path),
        "-db",
        str(db_prefix),
        "-task",
        BLAST_TASK,
        "-evalue",
        str(EVALUE),
        "-max_target_seqs",
        str(MAX_TARGET_SEQS),
        "-max_hsps",
        str(MAX_HSPS),
        "-num_threads",
        str(NUM_THREADS),
        "-outfmt",
        "6 qseqid sseqid bitscore length pident evalue",
    ]

    result = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )

    blast_tsv_path.write_text(
        result.stdout,
        encoding="utf-8",
    )


def load_blast_table(
    blast_tsv_path: Path,
) -> list[tuple[str, str, float]]:
    hits: list[tuple[str, str, float]] = []

    with blast_tsv_path.open() as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")

            if len(parts) < 3:
                continue

            query_id = parts[0]
            subject_id = parts[1]
            bitscore = float(parts[2])

            hits.append(
                (
                    query_id,
                    subject_id,
                    bitscore,
                )
            )

    return hits


def build_bitscore_matrix(
    ids: list[str],
    hits: list[tuple[str, str, float]],
) -> np.ndarray:
    index = {
        name: i
        for i, name in enumerate(ids)
    }

    n = len(ids)
    matrix = np.zeros(
        (n, n),
        dtype=float,
    )

    for query_id, subject_id, bitscore in hits:
        if query_id not in index or subject_id not in index:
            continue

        i = index[query_id]
        j = index[subject_id]

        if bitscore > matrix[i, j]:
            matrix[i, j] = bitscore

    matrix = 0.5 * (matrix + matrix.T)

    return matrix


def extract_self_scores(
    bitscore_matrix: np.ndarray,
) -> np.ndarray:
    self_scores = np.diag(bitscore_matrix).copy()
    self_scores[self_scores <= 0] = EPS

    return self_scores


def normbit_arithmetic_mean(
    bitscore_matrix: np.ndarray,
) -> np.ndarray:
    self_scores = extract_self_scores(bitscore_matrix)

    denominator = 0.5 * (
        self_scores[:, None]
        + self_scores[None, :]
    )

    similarity = bitscore_matrix / np.maximum(
        denominator,
        EPS,
    )

    similarity = np.clip(
        similarity,
        0.0,
        1.0,
    )

    np.fill_diagonal(
        similarity,
        1.0,
    )

    similarity = 0.5 * (
        similarity
        + similarity.T
    )

    return similarity


def normbit_geometric_mean(
    bitscore_matrix: np.ndarray,
) -> np.ndarray:
    self_scores = extract_self_scores(bitscore_matrix)

    denominator = np.sqrt(
        np.maximum(
            self_scores[:, None]
            * self_scores[None, :],
            EPS,
        )
    )

    similarity = bitscore_matrix / np.maximum(
        denominator,
        EPS,
    )

    similarity = np.clip(
        similarity,
        0.0,
        1.0,
    )

    np.fill_diagonal(
        similarity,
        1.0,
    )

    similarity = 0.5 * (
        similarity
        + similarity.T
    )

    return similarity


def similarity_to_log_distance(
    similarity: np.ndarray,
) -> np.ndarray:
    similarity = np.clip(
        np.asarray(
            similarity,
            dtype=float,
        ),
        0.0,
        1.0,
    )

    distance = -np.log(
        np.maximum(
            similarity,
            EPS,
        )
    )

    np.fill_diagonal(
        distance,
        0.0,
    )

    distance = 0.5 * (
        distance
        + distance.T
    )

    return distance


def save_pickle(
    obj,
    output_path: Path,
) -> None:
    with output_path.open("wb") as f:
        pickle.dump(
            obj,
            f,
        )


def save_npz(
    output_path: Path,
    **arrays,
) -> None:
    np.savez_compressed(
        output_path,
        **arrays,
    )


def save_matrix_manifest(
    rows: list[MatrixRow],
    output_path: Path,
) -> None:
    fieldnames = list(MatrixRow.__annotations__.keys())
    path_fields = {
        "fasta_path",
        "blast_tsv_path",
        "pickle_path",
        "npz_path",
    }
    root = PROJECT_ROOT.resolve()

    with output_path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )
        writer.writeheader()

        for row in rows:
            record = asdict(row)

            for field in path_fields:
                value = record.get(field)

                if not value:
                    continue

                path = Path(value).expanduser().resolve()

                try:
                    record[field] = (
                        path.relative_to(root).as_posix()
                    )
                except ValueError as error:
                    raise ValueError(
                        f"{field} is outside project root: {path}"
                    ) from error

            writer.writerow(record)


def process_one_dataset(
    tag: str,
    fasta_path: Path,
) -> MatrixRow:
    ids = read_fasta_ids(fasta_path)

    blast_tsv_path = MATRIX_DIR / f"{tag}.blast.tsv"
    pickle_path = MATRIX_DIR / f"{tag}_matrices.pkl"
    npz_path = MATRIX_DIR / f"{tag}_matrices.npz"
    db_prefix = BLAST_DB_DIR / tag

    row = MatrixRow(
        tag=tag,
        fasta_path=str(fasta_path),
        n_sequences=len(ids),
        blast_tsv_path=str(blast_tsv_path),
        pickle_path=str(pickle_path),
        npz_path=str(npz_path),
        status="planned",
        message="",
    )

    if (
        pickle_path.exists()
        and npz_path.exists()
        and not OVERWRITE
    ):
        return replace(
            row,
            status="ok",
            message="skipped_existing",
        )

    try:
        make_blast_db(
            fasta_path=fasta_path,
            db_prefix=db_prefix,
        )

        run_self_blast(
            fasta_path=fasta_path,
            db_prefix=db_prefix,
            blast_tsv_path=blast_tsv_path,
        )

        hits = load_blast_table(
            blast_tsv_path
        )

        bitscore_raw = build_bitscore_matrix(
            ids=ids,
            hits=hits,
        )

        paper_normbit_mean = normbit_arithmetic_mean(
            bitscore_raw
        )

        normbit_geom = normbit_geometric_mean(
            bitscore_raw
        )

        log_distance_from_mean = similarity_to_log_distance(
            paper_normbit_mean
        )

        log_distance_from_geom = similarity_to_log_distance(
            normbit_geom
        )

        payload = {
            "tag": tag,
            "ids": ids,
            "bitscore_raw": bitscore_raw,
            "paper_normbit_mean": paper_normbit_mean,
            "normbit_geom": normbit_geom,
            "log_distance_from_mean": log_distance_from_mean,
            "log_distance_from_geom": log_distance_from_geom,
        }

        save_pickle(
            payload,
            pickle_path,
        )

        save_npz(
            npz_path,
            bitscore_raw=bitscore_raw,
            paper_normbit_mean=paper_normbit_mean,
            normbit_geom=normbit_geom,
            log_distance_from_mean=log_distance_from_mean,
            log_distance_from_geom=log_distance_from_geom,
        )

    except Exception as error:
        return replace(
            row,
            status="failed",
            message=str(error)[:2000],
        )

    return replace(
        row,
        status="ok",
        message="",
    )

In [ ]:
ensure_directories()
check_blast_executables()

simulation_rows = read_simulation_manifest()

matrix_rows: list[MatrixRow] = []

for index, sim_row in enumerate(simulation_rows, start=1):
    tag = sim_row["tag"]
    fasta_path = resolve_fasta_path(sim_row)

    print(
        f"[{index:04d}/{len(simulation_rows):04d}] "
        f"{tag}"
    )

    matrix_rows.append(
        process_one_dataset(
            tag=tag,
            fasta_path=fasta_path,
        )
    )

matrix_manifest_path = (
    MATRIX_DIR
    / "matrix_manifest.csv"
)

save_matrix_manifest(
    matrix_rows,
    matrix_manifest_path,
)

status_counts = Counter(
    row.status
    for row in matrix_rows
)

print()
print("Similarity matrix construction completed.")

for status, count in sorted(status_counts.items()):
    print(f"{status}: {count}")

print()
print(f"Matrix manifest: {matrix_manifest_path}")